# Creating a bubble plot visualization to demonstrate parameter values

In [1]:
import os
from tempfile import template

import numpy as np
import pandas as pd
from rpy2.robjects.lib.grid import xaxis, yaxis
from scipy.stats import t
from pyreadr import read_r
import plotly.graph_objects as go
from sympy.printing.pretty.pretty_symbology import line_width

from shared.wald_test import *

In [2]:
PARAMETER_ESTIMATES_PATH = 'params/'

affective_motives = np.array(sorted([

    ### Affective motive
    'conversationalist', 'how_enjoyable', 'i_like_you', 'i_felt_close_to_my_partner', 'i_think_your_status', 'i_would_like_to_become_friends',
    # 'overall_affect', 'conv_leader', 'best_affect', 'responsive_1','responsive_2','responsive_3',
    'you_are_good_listener', 'you_are_kind', 'you_are_quickwitted','you_are_warm','you_like_me',

    #test affectives!
    'you_are_intelligent', 'you_are_friendly',

]), dtype=object)
cognitive_motives = np.array(sorted([

    ### Cognitive motives
    'anticipated_each_other_sr6', 'developed_joint_perspective_sr2', 'thoughts_became_more_alike_sr5',
    # 'in_common',

    #test cognitives!
    'shared_reality', 'our_thoughts_synced_up_sr1', 'saw_world_in_same_way_sr8',

]), dtype=object)

In [3]:
renamed_affective = [
    '_'.join([w for w in label.split('_') if len(w) > 1])
    for label in affective_motives
]

renamed_cognitive = [
    '_'.join([w for w in label.split('_') if len(w) > 1])
    for label in cognitive_motives
]

In [4]:
# color_choices = ['#6D8196', '#4A4A4A']
# color_choices = ['#6D8196', '#000000']
color_choices = ['#FFA896', '#38000A']
minimum_bubble_size = 2
scale_params_by = 1000
maximum_bubble_size = 40

from_last_k = 500
resample_n = 50
# resample_n = from_last_k
vis_dif = .1

#### Create data structure

In [5]:
affective_betas, affective_statistics = [], []

In [6]:
for f in affective_motives:
    data = read_r(os.path.join(PARAMETER_ESTIMATES_PATH, '{}.RData'.format(f)))

    phi = data['phi'].values.reshape(-1)[-from_last_k:]
    beta = data['beta'].values.reshape(-1)[-from_last_k:]

    phi = np.random.choice(phi, size=(resample_n,), replace=False)
    beta = np.random.choice(beta, size=(resample_n,), replace=False)

    std_phi = phi.std()
    std_beta = beta.std()

    phi_tstat = phi.mean() / (std_phi/np.sqrt(phi.shape[0]-1))
    beta_tstat = beta.mean() / (std_beta/np.sqrt(phi.shape[0]-1))

    dof = phi.shape[0] - 1

    affective_betas += [
        {
            'var': f,
            'phi': data['phi'].mean().item(),
            'beta': data['beta'].mean().item(),
            'std(phi)': std_phi,
            'std(beta)': std_beta,
            'dof': dof,
            't(phi)': phi_tstat.item(),
            'p(phi)': (1-t.cdf(np.abs(phi_tstat), df=dof)).item()*2,
            't(beta)': beta_tstat.item(),
            'p(beta)': (1-t.cdf(np.abs(beta_tstat), df=dof)).item()*2
        }
    ]

affective_betas = pd.DataFrame(affective_betas)

In [7]:
affective_betas.head(n=20)

,var,phi,beta,std(phi),std(beta),dof,t(phi),p(phi),t(beta),p(beta)
0,conversationalist,0.001188,-0.000434,0.000528,0.000822,49,15.044213,0.000000e+00,-4.454311,0.000049
1,how_enjoyable,-0.055549,-0.002025,0.113771,0.003939,49,-4.480115,4.478820e-05,-1.931345,0.059235
2,i_felt_close_to_my_partner,-0.001842,-0.030290,0.005003,0.126775,49,-0.355548,7.237051e-01,-2.324963,0.024260
3,i_like_you,-0.190803,-1.919631,0.230916,0.287331,49,-5.785852,4.994421e-07,-49.172741,0.000000
4,i_think_your_status,-0.211332,-1.137667,0.296737,0.665431,49,-4.075601,1.677047e-04,-12.462043,0.000000
5,i_would_like_to_become_friends,-0.005403,-0.000948,0.003261,0.001365,49,-9.392939,1.543210e-12,-5.191346,0.000004
6,you_are_friendly,-0.402568,-2.122478,0.298770,0.142244,49,-9.575585,8.324452e-13,-105.706025,0.000000
7,you_are_good_listener,-0.701679,-1.798813,0.403350,0.262706,49,-11.659589,8.881784e-16,-46.715489,0.000000
8,you_are_intelligent,-0.966169,-2.487884,0.318794,0.112274,49,-19.999028,0.000000e+00,-154.753227,0.000000
9,you_are_kind,-1.261589,-2.163427,0.164354,0.100467,49,-53.545973,0.000000e+00,-151.478569,0.000000


In [8]:
cognitive_betas, cognitive_statistics = [], []

In [9]:
for f in cognitive_motives:
    data = read_r(os.path.join(PARAMETER_ESTIMATES_PATH, '{}.RData'.format(f)))

    phi = data['phi'].values.reshape(-1)[-from_last_k:]
    beta = data['beta'].values.reshape(-1)[-from_last_k:]

    phi = np.random.choice(phi, size=(resample_n,), replace=False)
    beta = np.random.choice(beta, size=(resample_n,), replace=False)

    std_phi = phi.std()
    std_beta = beta.std()

    phi_tstat = phi.mean() / (std_phi/np.sqrt(phi.shape[0]-1))
    beta_tstat = beta.mean() / (std_beta/np.sqrt(phi.shape[0]-1))

    dof = phi.shape[0] - 1

    cognitive_betas += [
        {
            'var': f,
            'phi': data['phi'].mean().item(),
            'beta': data['beta'].mean().item(),
            'std(phi)': std_phi,
            'std(beta)': std_beta,
            'dof': dof,
            't(phi)': phi_tstat.item(),
            'p(phi)': (1-t.cdf(np.abs(phi_tstat), df=dof)).item()*2,
            't(beta)': beta_tstat.item(),
            'p(beta)': (1-t.cdf(np.abs(beta_tstat), df=dof)).item()*2
        }
    ]

cognitive_betas = pd.DataFrame(cognitive_betas)

In [10]:
cognitive_betas.head(n=20)

,var,phi,beta,std(phi),std(beta),dof,t(phi),p(phi),t(beta),p(beta)
0,anticipated_each_other_sr6,0.001401,0.000051,0.000615,0.000687,49,16.652529,0.000000e+00,-0.533794,5.958973e-01
1,developed_joint_perspective_sr2,-0.834020,-1.698003,0.613534,0.272174,49,-9.323298,1.954659e-12,-44.438111,0.000000e+00
2,our_thoughts_synced_up_sr1,0.000527,-0.001253,0.001127,0.002543,49,5.229233,3.508540e-06,-4.814865,1.455236e-05
3,saw_world_in_same_way_sr8,-0.318768,-0.967349,0.465752,0.507435,49,-5.280286,2.938766e-06,-13.172847,0.000000e+00
4,shared_reality,0.000146,-0.001065,0.000964,0.001044,49,1.107292,2.735740e-01,-8.732488,1.481770e-11
5,thoughts_became_more_alike_sr5,-0.049371,-1.522711,0.135989,0.294133,49,-2.326089,2.419452e-02,-35.860116,0.000000e+00


#### Merging docs

In [11]:
affective_betas['motive'] = 'affective'
cognitive_betas['motive'] = 'cognitive'

In [12]:
dataset = pd.concat([affective_betas,cognitive_betas], ignore_index=True)
dataset.head(n=30)

,var,phi,beta,std(phi),std(beta),dof,t(phi),p(phi),t(beta),p(beta),motive
0,conversationalist,0.001188,-0.000434,0.000528,0.000822,49,15.044213,0.000000e+00,-4.454311,4.878886e-05,affective
1,how_enjoyable,-0.055549,-0.002025,0.113771,0.003939,49,-4.480115,4.478820e-05,-1.931345,5.923535e-02,affective
2,i_felt_close_to_my_partner,-0.001842,-0.030290,0.005003,0.126775,49,-0.355548,7.237051e-01,-2.324963,2.425980e-02,affective
3,i_like_you,-0.190803,-1.919631,0.230916,0.287331,49,-5.785852,4.994421e-07,-49.172741,0.000000e+00,affective
4,i_think_your_status,-0.211332,-1.137667,0.296737,0.665431,49,-4.075601,1.677047e-04,-12.462043,0.000000e+00,affective
5,i_would_like_to_become_friends,-0.005403,-0.000948,0.003261,0.001365,49,-9.392939,1.543210e-12,-5.191346,4.000669e-06,affective
6,you_are_friendly,-0.402568,-2.122478,0.298770,0.142244,49,-9.575585,8.324452e-13,-105.706025,0.000000e+00,affective
7,you_are_good_listener,-0.701679,-1.798813,0.403350,0.262706,49,-11.659589,8.881784e-16,-46.715489,0.000000e+00,affective
8,you_are_intelligent,-0.966169,-2.487884,0.318794,0.112274,49,-19.999028,0.000000e+00,-154.753227,0.000000e+00,affective
9,you_are_kind,-1.261589,-2.163427,0.164354,0.100467,49,-53.545973,0.000000e+00,-151.478569,0.000000e+00,affective


In [14]:
dataset.to_csv('/Users/zacharyrosen/Desktop/ICLASP_vars.csv', index=False, encoding='utf-8')

#### Saving csvs

In [ ]:
cognitive_betas.to_csv('cognitive-motives.csv', index=False, encoding='utf-8')
affective_betas.to_csv('affective-motives.csv', index=False, encoding='utf-8')

## Bubble plots

#### Cognitive Motives

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y = np.array([2]*len(cognitive_betas)),
        x = np.array(list(range(len(cognitive_betas)))),
        text = ['ɸ: {}'.format(np.format_float_scientific(v,precision=2)) for v in cognitive_betas['phi']],
        mode='markers',
        name='self',
        legendgroup='self',
        showlegend=False,
        marker=dict(
            size=cognitive_betas['phi'].values.__abs__()*scale_params_by,
            sizemode='area',
            sizeref=2*cognitive_betas['phi'].values.__abs__().max()*scale_params_by/(maximum_bubble_size**2),
            sizemin=minimum_bubble_size,
            color=[color_choices[int(i>0)] for i in cognitive_betas['phi']]
        )
    )
)

fig.add_trace(
    go.Scatter(
        y = np.array([1]*len(cognitive_betas)),
        x = np.array(list(range(len(cognitive_betas)))),
        text = ['β: {}'.format(np.format_float_scientific(v,precision=2)) for v in cognitive_betas['beta']],
        mode='markers',
        name='other',
        legendgroup='other',
        showlegend=False,
        marker=dict(
            size=cognitive_betas['beta'].values.__abs__()*scale_params_by,
            sizemode='area',
            sizeref=2*cognitive_betas['beta'].values.__abs__().max()*scale_params_by/(maximum_bubble_size**2),
            sizemin=minimum_bubble_size,
            color=[color_choices[int(i>0)] for i in cognitive_betas['beta']]
        )
    )
)

fig.update_layout(
    template='plotly_white',
    yaxis=dict(
        tickmode='array',
        tickvals=[1,2],
        ticktext=['other', 'self'],
    ),
    xaxis=dict(
        tickmode='array',
        tickvals=list(range(len(cognitive_betas))),
        ticktext=renamed_cognitive,
    ),
    height=260
)

fig.show()

In [ ]:
fig.write_html('cognitive-motives.html')

#### Affective Motives

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y = np.array([2]*len(affective_betas)),
        x = np.array(list(range(len(affective_betas)))),
        text = ['ɸ: {}'.format(np.format_float_scientific(v,precision=2)) for v in affective_betas['phi']],
        mode='markers',
        name='self',
        legendgroup='self',
        showlegend=False,
        marker=dict(
            size=affective_betas['phi'].values.__abs__()*scale_params_by,
            sizemode='area',
            sizeref=2*affective_betas['phi'].values.__abs__().max()*scale_params_by/(maximum_bubble_size**2),
            sizemin=minimum_bubble_size,
            color=[color_choices[int(i>0)] for i in affective_betas['phi']]
        )
    )
)

fig.add_trace(
    go.Scatter(
        y = np.array([1]*len(affective_betas)),
        x = np.array(list(range(len(affective_betas)))),
        text = ['β: {}'.format(np.format_float_scientific(v,precision=2)) for v in affective_betas['beta']],
        mode='markers',
        name='other',
        legendgroup='other',
        showlegend=False,
        marker=dict(
            size=affective_betas['beta'].values.__abs__()*scale_params_by,
            sizemode='area',
            sizeref=2*affective_betas['beta'].values.__abs__().max()*scale_params_by/(maximum_bubble_size**2),
            sizemin=minimum_bubble_size,
            color=[color_choices[int(i>0)] for i in affective_betas['beta']]
        )
    )
)

fig.update_layout(
    template='plotly_white',
    yaxis=dict(
        tickmode='array',
        tickvals=[1,2],
        ticktext=['other', 'self'],
    ),
    xaxis=dict(
        tickmode='array',
        tickvals=list(range(len(affective_betas))),
        ticktext=renamed_affective,
    ),
    height=300
)

fig.show()

In [ ]:
fig.write_html('affective-motives.html')

## Lolipop plots

In [15]:
# colors = [
#     'purple',
#     'blue'
# ]

colors = [
    '#005587',
    '#FFB81C'
]

### Other

In [16]:
fig = go.Figure()

# abs_limit = (np.abs(dataset['beta'] / dataset['std(beta)']) < 5)
abs_limit = np.array([True]*len(dataset))
ct = 0
for j,motive_type in enumerate(dataset['motive'].unique()):
    sel = (dataset['motive'] == motive_type) & abs_limit
    sub = dataset.loc[sel]

    showlegend_at = sub.loc[(sub['p(beta)'] < .05)].index.values
    if len(showlegend_at):
        showlegend_at = showlegend_at[0]
    else:
        showlegend_at = sub.index.values[0]

    showlegend = True
    for i in sub.index:
        ct+=1
        showlegend = (i==showlegend_at).item()
        length_of_marker = dataset['beta'].loc[i] / dataset['std(beta)'].loc[i]

        ## line
        fig.add_trace(
            go.Scatter(
                x=sorted([0,length_of_marker]),
                y=[ct,ct],
                mode='lines',
                name=motive_type+'_line',
                legendgroup=motive_type,
                showlegend=False,
                line=dict(
                    width=3,
                    color=colors[j]
                ),
            )
        )

        ## marker
        if (dataset['p(beta)'].loc[i] > .05):
            fig.add_trace(
                go.Scatter(
                    x=[length_of_marker],
                    y=[ct],
                    mode='markers',
                    name=motive_type,
                    legendgroup=motive_type,
                    showlegend=showlegend,#showlegend,
                    marker=dict(
                        size=15,
                        line=dict(
                            width=3*(dataset['p(beta)'].loc[i] > .05),
                            color=colors[j]
                        ),
                        color='white'
                    )
                )
            )
        else:
            fig.add_trace(
                go.Scatter(
                    x=[length_of_marker],
                    y=[ct],
                    mode='markers',
                    name=motive_type,
                    legendgroup=motive_type,
                    showlegend=showlegend,#showlegend,
                    marker=dict(
                        size=15,
                        line=dict(
                            width=3*(dataset['p(beta)'].loc[i] > .05),
                            color=colors[j]
                        ),
                        color=colors[j]
                    )
                )
            )

fig.add_vline(x=0, layer='below', line_width=3)
fig.update_yaxes(
    tickmode='array',
    tickvals=list(range(1,abs_limit.sum()+1)),
    ticktext=dataset['var'].loc[abs_limit].values
)
# fig.update_xaxes(range=[(-dataset['beta'].__abs__().max())-vis_dif,dataset['beta'].__abs__().max()+vis_dif])
fig.update_layout(
    template='ygridoff',
    # width=600,
    margin=dict(l=225,r=10),
    xaxis=dict(automargin=True)
)
fig.show()

In [17]:
# fig.write_html('lolipop-other.html')
# fig.write_html('lolipop-other-all.html')
fig.write_html('/Users/zacharyrosen/Desktop/lolipop-other-all.html')

##### old code...

In [ ]:
# fig = go.Figure()
#
# ct = 0
# for j,motive_type in enumerate(dataset['motive'].unique()):
#     sel = (dataset['motive']==motive_type) & (dataset['beta'].__abs__() <= .0027)
#     sub = dataset.loc[sel]
#
#     showlegend_at = sub.loc[(sub['p(beta)'] < .05)].index.values
#     if len(showlegend_at):
#         showlegend_at = showlegend_at[0]
#     else:
#         showlegend_at = sub.index.values[0]
#
#     showlegend = True
#     for i in sub.index:
#         ct+=1
#         showlegend = (i==showlegend_at).item()
#         length_of_marker = dataset['beta'].loc[i] / dataset['std(beta)'].loc[i]
#
#         ## line
#         fig.add_trace(
#             go.Scatter(
#                 x=sorted([0,length_of_marker]),
#                 y=[ct,ct],
#                 mode='lines',
#                 name=motive_type+'_line',
#                 legendgroup=motive_type,
#                 showlegend=False,
#                 line=dict(
#                     width=3,
#                     color=colors[j]
#                 ),
#             )
#         )
#
#         ## marker
#         if (dataset['p(beta)'].loc[i] > .05):
#             fig.add_trace(
#                 go.Scatter(
#                     x=[length_of_marker],
#                     y=[ct],
#                     mode='markers',
#                     name=motive_type,
#                     legendgroup=motive_type,
#                     showlegend=showlegend,#showlegend,
#                     marker=dict(
#                         size=15,
#                         line=dict(
#                             width=3*(dataset['p(beta)'].loc[i] > .05),
#                             color=colors[j]
#                         ),
#                         color='white'
#                     )
#                 )
#             )
#         else:
#             fig.add_trace(
#                 go.Scatter(
#                     x=[length_of_marker],
#                     y=[ct],
#                     mode='markers',
#                     name=motive_type,
#                     legendgroup=motive_type,
#                     showlegend=showlegend,#showlegend,
#                     marker=dict(
#                         size=15,
#                         line=dict(
#                             width=3*(dataset['p(beta)'].loc[i] > .05),
#                             color=colors[j]
#                         ),
#                         color=colors[j]
#                     )
#                 )
#             )
#
# fig.add_vline(x=0, layer='below', line_width=3)
# fig.update_yaxes(
#     tickmode='array',
#     tickvals=list(range(1,ct+1)),
#     ticktext=dataset['var'].values
# )
# # fig.update_xaxes(range=[-.003,.003])
# fig.update_layout(
#     template='ygridoff',
#     # width=600,
#     margin=dict(l=225,r=10),
#     xaxis=dict(automargin=True)
# )
# fig.show()

In [ ]:
# fig.write_html('lolipop-other-zoomed.html')
# # fig.write_html('/Users/zacharyrosen/Desktop/lolipop-other-2.html')

### Self

In [ ]:
fig = go.Figure()

# abs_limit = np.array([True]*len(dataset))#(np.abs(dataset['phi'] / dataset['std(phi)']) < 5)
ct = 0
for j,motive_type in enumerate(dataset['motive'].unique()):
    sel = (dataset['motive'] == motive_type) & abs_limit
    sub = dataset.loc[sel]

    showlegend_at = sub.loc[(sub['p(phi)'] < .05)].index.values
    if len(showlegend_at):
        showlegend_at = showlegend_at[0]
    else:
        showlegend_at = sub.index.values[0]

    showlegend = True
    for i in sub.index:
        ct+=1
        showlegend = (i==showlegend_at).item()
        length_of_marker = dataset['phi'].loc[i] / dataset['std(phi)'].loc[i]

        ## line
        fig.add_trace(
            go.Scatter(
                x=sorted([0,length_of_marker]),
                y=[ct,ct],
                mode='lines',
                name=motive_type+'_line',
                legendgroup=motive_type,
                showlegend=False,
                line=dict(
                    width=3,
                    color=colors[j]
                ),
            )
        )

        ## marker
        if (dataset['p(phi)'].loc[i] > .05):
            fig.add_trace(
                go.Scatter(
                    x=[length_of_marker],
                    y=[ct],
                    mode='markers',
                    name=motive_type,
                    legendgroup=motive_type,
                    showlegend=showlegend,#showlegend,
                    marker=dict(
                        size=15,
                        line=dict(
                            width=3*(dataset['p(phi)'].loc[i] > .05),
                            color=colors[j]
                        ),
                        color='white'
                    )
                )
            )
        else:
            fig.add_trace(
                go.Scatter(
                    x=[length_of_marker],
                    y=[ct],
                    mode='markers',
                    name=motive_type,
                    legendgroup=motive_type,
                    showlegend=showlegend,#showlegend,
                    marker=dict(
                        size=15,
                        line=dict(
                            width=3*(dataset['p(phi)'].loc[i] > .05),
                            color=colors[j]
                        ),
                        color=colors[j]
                    )
                )
            )

fig.add_vline(x=0, layer='below', line_width=3)
fig.update_yaxes(
    tickmode='array',
    tickvals=list(range(1,abs_limit.sum()+1)),
    ticktext=dataset['var'].loc[abs_limit].values
)
# fig.update_xaxes(range=[(-dataset['phi'].__abs__().max())-vis_dif,dataset['phi'].__abs__().max()+vis_dif])
fig.update_layout(
    template='ygridoff',
    # width=600,
    margin=dict(l=225,r=10),
    xaxis=dict(automargin=True)
)
fig.show()

In [ ]:
fig.write_html('lolipop-self.html')

##### old code...

In [ ]:
# fig = go.Figure()
#
# ct = 0
# for j,motive_type in enumerate(dataset['motive'].unique()):
#     sel = (dataset['motive']==motive_type) & (dataset['phi'].__abs__() <= .004)
#     sub = dataset.loc[sel]
#
#     showlegend_at = sub.loc[(sub['p(phi)'] < .05)].index.values
#     if len(showlegend_at):
#         showlegend_at = showlegend_at[0]
#     else:
#         showlegend_at = sub.index.values[0]
#
#     for i in sub.index:
#         ct+=1
#         showlegend = (i==showlegend_at).item()
#         length_of_marker = dataset['phi'].loc[i] / dataset['std(phi)'].loc[i]
#
#         ## line
#         fig.add_trace(
#             go.Scatter(
#                 x=sorted([0,length_of_marker]),
#                 y=[ct,ct],
#                 mode='lines',
#                 name=motive_type+'_line',
#                 legendgroup=motive_type,
#                 showlegend=False,
#                 line=dict(
#                     width=3,
#                     color=colors[j]
#                 ),
#             )
#         )
#
#         ## marker
#         if (dataset['p(phi)'].loc[i] > .05):
#             fig.add_trace(
#                 go.Scatter(
#                     x=[length_of_marker],
#                     y=[ct],
#                     mode='markers',
#                     name=motive_type,
#                     legendgroup=motive_type,
#                     showlegend=showlegend,#showlegend,
#                     marker=dict(
#                         size=15,
#                         line=dict(
#                             width=3*(dataset['p(phi)'].loc[i] > .05),
#                             color=colors[j]
#                         ),
#                         color='white'
#                     )
#                 )
#             )
#         else:
#             fig.add_trace(
#                 go.Scatter(
#                     x=[length_of_marker],
#                     y=[ct],
#                     mode='markers',
#                     name=motive_type,
#                     legendgroup=motive_type,
#                     showlegend=showlegend,#showlegend,
#                     marker=dict(
#                         size=15,
#                         line=dict(
#                             width=3*(dataset['p(phi)'].loc[i] > .05),
#                             color=colors[j]
#                         ),
#                         color=colors[j]
#                     )
#                 )
#             )
#
# fig.add_vline(x=0, layer='below', line_width=3)
# fig.update_yaxes(
#     tickmode='array',
#     tickvals=list(range(1,ct+1)),
#     ticktext=dataset['var'].values
# )
# # fig.update_xaxes(range=[-.0045,.0045])
# fig.update_layout(
#     template='ygridoff',
#     # width=600,
#     margin=dict(l=225,r=10),
#     xaxis=dict(automargin=True)
# )
# fig.show()

In [ ]:
# fig.write_html('lolipop-self-zoomed.html')
# # fig.write_html('/Users/zacharyrosen/Desktop/lolipop-other-2.html')

### Hypothesis examples

In [ ]:
colors = [
    '#1d3c45',
    '#d2601a'
]

##### Lollipop

In [ ]:
fig = go.Figure()

In [ ]:
# faster
fig.add_trace(
    go.Scatter(
        x=[-1,0],
        y=[1,1],
        mode='lines',
        name='fast changes'+'_line',
        legendgroup='fast changes',
        showlegend=False,
        line=dict(
            width=3,
            color=colors[0]
        ),
    )
)
fig.add_trace(
    go.Scatter(
        x=[-1],
        y=[1],
        mode='markers',
        name='fast changes',
        legendgroup='fast changes',
        showlegend=True,#showlegend,
        marker=dict(
            size=15,
            line=dict(
                width=3*(dataset['p(beta)'].loc[i] > .05),
                color=colors[0]
            ),
            color='white'
        )
    )
)

# slower
fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[-1,-1],
        mode='lines',
        name='slow changes'+'_line',
        legendgroup='slow changes',
        showlegend=False,
        line=dict(
            width=3,
            color=colors[1]
        ),
    )
)
fig.add_trace(
    go.Scatter(
        x=[1],
        y=[-1],
        mode='markers',
        name='slow changes',
        legendgroup='slow changes',
        showlegend=True,#showlegend,
        marker=dict(
            size=15,
            line=dict(
                width=3,
                color=colors[1]
            ),
            color='white'
        )
    )
)

In [ ]:
fig.update_layout(
    template='ygridoff',
    legend=dict(
        x=0,
        y=1,
        xanchor='left',  # Anchors the left side of the legend box to the x position
        yanchor='top'    # Anchors the top side of the legend box to the y position
    ),
    # Optional: adjust margins if the legend overlaps the plot title or top area
    margin=dict(t=100),
    yaxis=dict(showticklabels=False)
)
fig.show()

In [ ]:
fig.write_html('example_hypotheses/hypthosis-lollipop.html')

##### exponential growth

In [ ]:
time = np.linspace(1,50,100)

Fast rate of change

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=time,
        y=time**(-2),
        mode='lines',
        line=dict(
            color=colors[0],
            width=10
        )
    )
)

In [ ]:
fig.update_layout(
    template='ygridoff',
    yaxis=dict(showticklabels=False)
)
fig.show()

In [ ]:
fig.write_html('example_hypotheses/curve-fast.html')

Slow rate of change

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=time,
        y=time**(15),
        mode='lines',
        line=dict(
            color=colors[1],
            width=10
        )
    )
)

In [ ]:
fig.update_layout(
    template='ygridoff',
    yaxis=dict(showticklabels=False)
)
fig.show()

In [ ]:
fig.write_html('example_hypotheses/curve-slow.html')

## Fast Divergence versus Durative Convergence

### Load CANDOR metadata (?)

In [ ]:
CANDOR_META_DATA_PATH = '../../ComplexityScienceOfEverydayConvos/english/data/all-candor-meta_data.csv'
os.path.exists(CANDOR_META_DATA_PATH)

In [ ]:
df = pd.read_csv(CANDOR_META_DATA_PATH)[['user_id', 'convo_id']+affective_motives.tolist()+cognitive_motives.tolist()]
df.head()

### T-test and cluster

In [18]:
from scipy.stats import t

In [19]:
def tstat(x1,x2,s1,s2,k1,k2):
    varo, varn = s1**2, s2**2
    svaro, svarn = varo/k1, varn/k2
    svaro_svarn = svaro + svarn

    tstat = (x1-x2) / np.sqrt(svaro_svarn)

    dof_denom = ((svaro**2) / (k1-1)) + ((svarn**2) / (k2-1))
    dof = (svaro_svarn**2) / dof_denom

    return tstat, dof

def Pt(x1,x2,s1,s2,k1,k2):
    ts, dof = tstat(x1,x2,s1,s2,k1,k2)
    return t.sf(ts.__abs__(), dof)

In [20]:
minx, mins = dataset[['beta', 'std(beta)']].loc[
    (dataset['beta'] == dataset['beta'].loc[dataset['p(beta)'] < .01].min())
].values[0]
minx, mins

(np.float64(-2.4878840347126623), np.float64(0.11227438843986522))

In [21]:
maxx, maxs = dataset[['beta', 'std(beta)']].loc[
    dataset['beta']==dataset['beta'].loc[dataset['p(beta)'] < .01].max()
].values[0]
maxx, maxs

(np.float64(-0.00043431492462509165), np.float64(0.000821574483741135))

In [22]:
dataset['min_pole_p'] = [Pt(x,minx,sd,mins,50,50) for x,sd in dataset[['beta', 'std(beta)']].values]
dataset['max_pole_p'] = [Pt(x,maxx,sd,maxs,50,50) for x,sd in dataset[['beta', 'std(beta)']].values]
dataset['normalizer'] = dataset[['min_pole_p', 'max_pole_p']].values.sum(axis=-1)


dataset[['min_pole_p', 'max_pole_p']] = dataset[['min_pole_p', 'max_pole_p']].values / dataset['normalizer'].values.reshape(-1,1)

In [23]:
dataset[['var', 'beta', 'min_pole_p', 'max_pole_p', 'normalizer']].head(n=20)

,var,beta,min_pole_p,max_pole_p,normalizer
0,conversationalist,-0.000434,7.678017e-68,1.000000e+00,5.000000e-01
1,how_enjoyable,-0.002025,8.365134e-66,1.000000e+00,3.602704e-03
2,i_felt_close_to_my_partner,-0.030290,2.033390e-99,1.000000e+00,5.112435e-02
3,i_like_you,-1.919631,1.000000e+00,1.217519e-23,6.516370e-20
4,i_think_your_status,-1.137667,7.628462e-04,9.992372e-01,1.305788e-16
5,i_would_like_to_become_friends,-0.000948,3.009510e-66,1.000000e+00,1.260649e-02
6,you_are_friendly,-2.122478,1.000000e+00,4.804227e-35,1.977130e-25
7,you_are_good_listener,-1.798813,1.000000e+00,8.022390e-18,3.039261e-26
8,you_are_intelligent,-2.487884,1.000000e+00,7.678017e-68,5.000000e-01
9,you_are_kind,-2.163427,1.000000e+00,1.614685e-40,9.622495e-28


In [24]:
dataset.to_csv('/Users/zacharyrosen/Desktop/ICLASP-results.csv', index=False, encoding='utf-8')